In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
#from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3


In [ ]:
load_dotenv()
llm=ChatOpenAI(proxy_model_name='gpt-4o')

In [ ]:
class ChatState(TypedDict):
    messages=Annotated[list[BaseMessage],add_messages]


In [ ]:
def chat_node(state:ChatState):
    messages=state['messages']
    response=llm.invoke(messages)
    return{'messages':[response]}

In [ ]:
conn=sqlite3.connect(database='chatbot.db',check_same_thread=False)
#checkpointer
checkpointer=SqliteSaver(conn=conn)

In [9]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

NameError: name 'checkpointer' is not defined

In [10]:
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config['configurable']['thread_id'])

    return list(all_threads)